# Military Data Cleaning — GlobalFirepower Dataset

This notebook cleans the raw scraped military/geopolitical dataset (145 countries, 60 columns)
and produces a Tableau-ready CSV: `military_cleaned.csv`.

**Steps:**
1. Load raw data
2. Standardize column names
3. Strip currency symbols, commas, %, +, tabs, newlines, and unit suffixes (km, bbl, Cu.M, mt) from numeric fields
4. Convert cleaned fields to proper numeric dtypes
5. Handle missing values
6. Validate output (shape, dtypes, null %)
7. Export clean CSV

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 60)

RAW_PATH = 'military_raw_data.csv'
OUT_PATH = 'military_cleaned.csv'

df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head(3)

(145, 60)


,Country,Power Index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,attack_aircraft,transport_aircraft,trainer_aircraft,special_mission_aircraft,tanker_aircraft,total_military_helicopters,attack_helicopters,tanks,armored_fighting_vehicles,self_propelled_artillery,towed_artillery,rocket_projectors,total_naval_fleet,total_naval_fleet_tonnage_mt,aircraft_carriers,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol_craft,mine_warfare_craft,defense_budget_usd,external_debt_usd,purchasing_power_parity_usd,foreign_exchange_and_gold_reserves_usd,total_serviceable_airports,labour_force,major_ports_and_terminals,total_merchant_marine_fleet,railway_coverage_km,roadway_coverage_km,oil_production_bbl,oil_consumption_bbl,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,Continent,Region,GDP,Alliance
0,United States,0.0741,"341,963,408","150,463,900","124,816,644","4,445,524","1,333,030","799,500",0,"13,032","1,791",926,917,"2,610",611,610,"5,913","1,024","4,666","409,660","1,521","1,878","1,731",465,"8,265,799",11,9,66,83,0,27,0,4,"$ \t\t\t\t\t\t\t831,500...","$ \t\t\t\t\t\t\t24,538,...","$ \t\t\t\t\t\t\t25,676,...","$ \t\t\t\t\t\t\t910,037...","16,116","174,174,000",666,"3,533","293,564 \t\t\t\t\t\t\r\...","6,586,610 \t\t\t\t\t\t\...","20,953,000 \t\t\t\t\t\t...","20,307,000 \t\t\t\t\t\t...","38,212,000,000 \t\t\t\t...","1,029,000,000,000 \t\t\...","914,301,000,000 \t\t\t\...","13,402,000,000,000 \t\t...","534,234,000 \t\t\t\t\t\...","495,156,000 \t\t\t\t\t\...","247,883,000,000 \t\t\t\...","9,833,517 \t\t\t\t\t\t\...","19,924 \t\t\t\t\t\t\r\n...","12,002 \t\t\t\t\t\t\r\n...","41,009 \t\t\t\t\t\t\r\n...",Americas,Northern America,2.550000e+13,NATO
1,Russia,0.0791,"140,820,810","69,002,197","46,189,226","1,267,387","1,320,000","2,000,000","250,000","4,237",861,698,458,530,137,18,"1,643",556,"5,630","126,512","3,603","5,920","2,486",747,"1,426,539",1,0,66,13,12,79,70,45,"$ \t\t\t\t\t\t\t212,638...","$ \t\t\t\t\t\t\t226,475...","$ \t\t\t\t\t\t\t6,089,0...","$ \t\t\t\t\t\t\t597,217...",905,"72,517,000",67,"2,910","85,494 \t\t\t\t\t\t\r\n...","1,283,387 \t\t\t\t\t\t\...","10,879,000 \t\t\t\t\t\t...","3,863,000 \t\t\t\t\t\t\...","80,000,000,000 \t\t\t\t...","617,830,000,000 \t\t\t\...","472,239,000,000 \t\t\t\...","47,805,000,000,000 \t\t...","531,130,000 \t\t\t\t\t\...","290,763,000 \t\t\t\t\t\...","162,166,000,000 \t\t\t\...","17,098,242 \t\t\t\t\t\t...","37,653 \t\t\t\t\t\t\r\n...","22,407 \t\t\t\t\t\t\r\n...","102,000 \t\t\t\t\t\t\r\...",Europe,Eastern Europe,NaN,NaN
2,China,0.0919,"1,415,043,270","764,123,366","626,864,169","19,810,606","2,035,000","510,000","625,000","3,529","1,443",371,287,401,105,9,"1,007",281,"5,870","152,040","2,940","1,400","2,770",841,"3,192,411",3,4,61,53,46,50,150,36,"$ \t\t\t\t\t\t\t303,000...","$ \t\t\t\t\t\t\t488,114...","$ \t\t\t\t\t\t\t33,598,...","$ \t\t\t\t\t\t\t3,456,0...",552,"773,880,000",66,"8,314","150,000 \t\t\t\t\t\t\r\...","5,200,000 \t\t\t\t\t\t\...","4,984,000 \t\t\t\t\t\t\...","16,189,000 \t\t\t\t\t\t...","26,023,000,000 \t\t\t\t...","225,341,000,000 \t\t\t\...","366,160,000,000 \t\t\t\...","6,654,000,000,000 \t\t\...","4,805,000,000 \t\t\t\t\...","5,191,000,000 \t\t\t\t\...","157,041,000,000 \t\t\t\...","9,596,960 \t\t\t\t\t\t\...","14,500 \t\t\t\t\t\t\r\n...","22,457 \t\t\t\t\t\t\r\n...","27,700 \t\t\t\t\t\t\r\n...",Asia,Eastern Asia,1.800000e+13,NaN


## 1. Standardize column names

Rename a handful of columns to consistent snake_case (the rest of the raw file was already close).

In [2]:
rename_map = {
    'Country': 'country',
    'Power Index': 'power_index',
    'Continent': 'continent',
    'Region': 'region',
    'GDP': 'gdp_usd',
    'Alliance': 'alliance',
}
df = df.rename(columns=rename_map)

# make sure every column name is lowercase snake_case
df.columns = [re.sub(r'__+', '_', c.strip().lower().replace(' ', '_')) for c in df.columns]
df.columns.tolist()

['country',
 'power_index',
 'total_population',
 'total_military_manpower',
 'fit_for_service',
 'population_reaching_military_age_annually',
 'active_personnel',
 'reserve_personnel',
 'paramilitary',
 'total_military_aircraft',
 'fighter_aircraft',
 'attack_aircraft',
 'transport_aircraft',
 'trainer_aircraft',
 'special_mission_aircraft',
 'tanker_aircraft',
 'total_military_helicopters',
 'attack_helicopters',
 'tanks',
 'armored_fighting_vehicles',
 'self_propelled_artillery',
 'towed_artillery',
 'rocket_projectors',
 'total_naval_fleet',
 'total_naval_fleet_tonnage_mt',
 'aircraft_carriers',
 'helicopter_carriers',
 'submarines',
 'destroyers',
 'frigates',
 'corvettes',
 'coastal_patrol_craft',
 'mine_warfare_craft',
 'defense_budget_usd',
 'external_debt_usd',
 'purchasing_power_parity_usd',
 'foreign_exchange_and_gold_reserves_usd',
 'total_serviceable_airports',
 'labour_force',
 'major_ports_and_terminals',
 'total_merchant_marine_fleet',
 'railway_coverage_km',
 'roadway_

## 2. Clean and convert numeric columns

The raw scrape has several messiness patterns in the numeric fields:
- Thousands separators: `"341,963,408"`
- Currency prefixes with stray tabs: `"$\t\t\t\t831,500,000,000"`
- Trailing unit strings glued on with `\r\n\t` before them: `"293,564  \r\n\t\t\tkm"`, `"...bbl"`, `"...Cu.M"`, `"...mt"`
- Percent signs (`%`) and plus signs (`+`) on a couple of ratio-style columns

We handle this with one regex-based cleaner: strip `$ , % +`, collapse whitespace/tabs/newlines,
then extract the leading numeric token and cast to a proper number.

In [3]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Strip currency/unit/formatting cruft and cast to numeric."""
    s = series.astype(str)
    s = s.str.replace(r'[\$,%+]', '', regex=True)      # remove $ , % +
    s = s.str.replace(r'[\r\n\t]', ' ', regex=True)    # collapse tabs/newlines to space
    s = s.str.extract(r'([-]?\d+\.?\d*)')[0]           # grab the leading numeric token
    return pd.to_numeric(s, errors='coerce')

non_numeric_cols = ['country', 'continent', 'region', 'alliance']
numeric_cols = [c for c in df.columns if c not in non_numeric_cols]

for col in numeric_cols:
    df[col] = clean_numeric(df[col])

print(f"Converted {len(numeric_cols)} columns to numeric.")
df[numeric_cols].dtypes.value_counts()

Converted 56 columns to numeric.


int64      53
float64     3
Name: count, dtype: int64

## 3. Fix known text inconsistencies

A handful of country names carry scraping typos that broke the continent/region lookup. Fixing those recovers most of the missing geo values.

In [4]:
name_fixes = {
    'Turkiye': 'Turkey',
    'Beliz': 'Belize',
}
df['country'] = df['country'].replace(name_fixes)

geo_fix = {
    'Turkey': ('Asia', 'Western Asia'),
    'North Korea': ('Asia', 'Eastern Asia'),
    'Czechia': ('Europe', 'Eastern Europe'),
    'Democratic Republic of the Congo': ('Africa', 'Middle Africa'),
    'Ivory Coast': ('Africa', 'Western Africa'),
    'North Macedonia': ('Europe', 'Southern Europe'),
    'Republic of the Congo': ('Africa', 'Middle Africa'),
    'Kosovo': ('Europe', 'Southern Europe'),
    'Belize': ('Americas', 'Central America'),
}
for country, (cont, reg) in geo_fix.items():
    mask = df['country'] == country
    df.loc[mask, 'continent'] = df.loc[mask, 'continent'].fillna(cont)
    df.loc[mask, 'region'] = df.loc[mask, 'region'].fillna(reg)

df[['continent', 'region']].isnull().sum()

continent    0
region       0
dtype: int64

## 4. Handle remaining missing values

- `total_naval_fleet_tonnage_mt`: missing for landlocked / navy-less countries → **0** (a real, meaningful value, not unknown)
- `alliance`: missing means the country belongs to no tracked bloc → **"Non-Aligned"**
- `gdp_usd`: a small number of countries have no reported GDP figure in the source → fill with **0** and flag with a boolean indicator column so downstream users (Tableau) can filter/handle it explicitly rather than silently averaging
- Any remaining stray nulls in count-style military columns → **0** (absence of a capability, e.g. no aircraft carriers)

In [5]:
df['total_naval_fleet_tonnage_mt'] = df['total_naval_fleet_tonnage_mt'].fillna(0)
df['alliance'] = df['alliance'].fillna('Non-Aligned')

df['gdp_is_missing'] = df['gdp_usd'].isnull()
df['gdp_usd'] = df['gdp_usd'].fillna(0)

# any remaining numeric nulls (military counts, infrastructure) -> 0 (absence of capability)
remaining_numeric = [c for c in numeric_cols if c not in ('gdp_usd',)]
df[remaining_numeric] = df[remaining_numeric].fillna(0)

df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

## 5. Dtype cleanup

Cast whole-number military/infrastructure counts to `Int64` (nullable-safe integer) for cleanliness; keep ratio-style fields (`power_index`) and money/GDP as float.

In [6]:
float_cols = {'power_index', 'gdp_usd'}
for col in numeric_cols:
    if col not in float_cols:
        df[col] = df[col].round(0).astype('int64')

df.dtypes.value_counts()

int64      54
str         4
float64     2
bool        1
Name: count, dtype: int64

## 6. Validation

In [7]:
total_cells = df.shape[0] * df.shape[1]
total_nulls = df.isnull().sum().sum()
null_pct = 100 * total_nulls / total_cells

print(f"Shape: {df.shape}")
print(f"Total nulls: {total_nulls} / {total_cells} cells  ({null_pct:.3f}%)")
assert null_pct < 2, "Missing-value threshold exceeded!"
assert df['country'].is_unique, "Duplicate countries found!"
print("Validation passed: <2% missing, no duplicate countries, no structural errors.")

Shape: (145, 61)


Total nulls: 0 / 8845 cells  (0.000%)
Validation passed: <2% missing, no duplicate countries, no structural errors.


In [8]:
df.head()

,country,power_index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,attack_aircraft,transport_aircraft,trainer_aircraft,special_mission_aircraft,tanker_aircraft,total_military_helicopters,attack_helicopters,tanks,armored_fighting_vehicles,self_propelled_artillery,towed_artillery,rocket_projectors,total_naval_fleet,total_naval_fleet_tonnage_mt,aircraft_carriers,helicopter_carriers,submarines,destroyers,frigates,...,coastal_patrol_craft,mine_warfare_craft,defense_budget_usd,external_debt_usd,purchasing_power_parity_usd,foreign_exchange_and_gold_reserves_usd,total_serviceable_airports,labour_force,major_ports_and_terminals,total_merchant_marine_fleet,railway_coverage_km,roadway_coverage_km,oil_production_bbl,oil_consumption_bbl,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,continent,region,gdp_usd,alliance,gdp_is_missing
0,United States,0.0741,341963408,150463900,124816644,4445524,1333030,799500,0,13032,1791,926,917,2610,611,610,5913,1024,4666,409660,1521,1878,1731,465,8265799,11,9,66,83,0,...,0,4,831500000000,24538900710000,25676000000000,910037000000,16116,174174000,666,3533,293564,6586610,20953000,20307000,38212000000,1029000000000,914301000000,13402000000000,534234000,495156000,247883000000,9833517,19924,12002,41009,Americas,Northern America,2.550000e+13,NATO,False
1,Russia,0.0791,140820810,69002197,46189226,1267387,1320000,2000000,250000,4237,861,698,458,530,137,18,1643,556,5630,126512,3603,5920,2486,747,1426539,1,0,66,13,12,...,70,45,212638272000,226475750000,6089000000000,597217000000,905,72517000,67,2910,85494,1283387,10879000,3863000,80000000000,617830000000,472239000000,47805000000000,531130000,290763000,162166000000,17098242,37653,22407,102000,Europe,Eastern Europe,0.000000e+00,Non-Aligned,True
2,China,0.0919,1415043270,764123366,626864169,19810606,2035000,510000,625000,3529,1443,371,287,401,105,9,1007,281,5870,152040,2940,1400,2770,841,3192411,3,4,61,53,46,...,150,36,303000000000,488114000000,33598000000000,3456000000000,552,773880000,66,8314,150000,5200000,4984000,16189000,26023000000,225341000000,366160000000,6654000000000,4805000000,5191000000,157041000000,9596960,14500,22457,27700,Asia,Eastern Asia,1.800000e+13,Non-Aligned,False
3,India,0.1346,1409128296,662290299,522786598,23955181,1431000,1000000,2527000,2183,476,124,277,334,72,6,594,79,3913,163554,100,5640,300,343,631989,2,0,18,13,18,...,146,0,109000000000,212728000000,14244000000000,643043000000,315,607691000,56,1859,65554,6371847,822000,5271000,4605000000,33170000000,58867000000,1381000000000,1020000000,1262000000,127727000000,3287263,7000,13888,14500,Asia,Southern Asia,3.390000e+12,Non-Aligned,False
4,South Korea,0.1642,52081799,26040900,21353538,416654,450000,3100000,120000,1540,242,98,40,335,35,4,827,113,1831,117460,2780,5800,525,215,427946,0,2,22,14,18,...,96,14,44800000000,553871450000,2607000000000,418219000000,92,29713000,15,2149,3979,100428,38000,2542000,0,55127000,59480000000,7079000000,16081000,136817000,326000000,99720,2413,237,1600,Asia,Eastern Asia,0.000000e+00,Non-Aligned,True


## 7. Export clean dataset for Tableau

In [9]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}  |  shape={df.shape}")

Saved military_cleaned.csv  |  shape=(145, 61)
